# 02 — Evaluación de Forecast sobre el Catálogo

Este notebook permite correr y visualizar la evaluación masiva de forecast (horse-race de modelos) sobre el catálogo completo o una submuestra.

**Flujo:**
1. Configurar `EvalConfig` en la celda de setup
2. Correr la evaluación (o cargar un run existente)
3. Explorar métricas globales, por segmento y comparación entre runs

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from planning_core.repository import CanonicalRepository
from planning_core.services import PlanningService
from planning_core.forecasting.evaluation import (
    EvalConfig,
    run_catalog_evaluation,
    aggregator,
    comparator,
    run_store,
)

BASE_DIR = "../output/eval_runs"

print("Setup OK")

## 1 — Correr evaluación (o cargar run existente)

Ajusta `RUN_EXISTING_ID` para cargar un run ya guardado, o pon `None` para correr uno nuevo.

In [ ]:
# ── Configuración del experimento ────────────────────────────────────────────
CONFIG = EvalConfig(
    granularity = "M",
    h           = 3,
    n_windows   = 3,
    use_lgbm    = False,
    run_name    = "baseline",
    sample_n    = None,   # None = catálogo completo | int = submuestra
    random_seed = 42,
)

N_JOBS = 4           # 1 = secuencial | -1 = todos los CPUs
CHECKPOINT_EVERY = 50
RUN_EXISTING_ID = None  # ej: "20260325_143201_baseline" para cargar sin re-correr

# ── Cargar o correr ──────────────────────────────────────────────────────────
if RUN_EXISTING_ID is not None:
    result = run_store.load_run(RUN_EXISTING_ID, BASE_DIR)
    print(f"Run cargado: {result.run_id}  ({result.n_evaluated} SKUs)")
else:
    repo    = CanonicalRepository()
    service = PlanningService(repo)
    result = run_catalog_evaluation(
        service,
        CONFIG,
        verbose=True,
        n_jobs=N_JOBS,
        checkpoint_every=CHECKPOINT_EVERY,
    )
    run_dir = run_store.save_run(result, BASE_DIR)
    print(f"\nGuardado en: {run_dir}")

df = result.sku_results
print(f"\nShape: {df.shape}  |  Columnas: {list(df.columns)}")

## 2 — Métricas globales

In [7]:
global_m = aggregator.compute_global_metrics(df)

summary = pd.DataFrame([
    {"Métrica": "SKUs evaluados",  "Valor": global_m["n_total"]},
    {"Métrica": "OK",              "Valor": global_m["n_ok"]},
    {"Métrica": "Fallback",        "Valor": global_m["n_fallback"]},
    {"Métrica": "No forecast",     "Valor": global_m["n_no_forecast"]},
    {"Métrica": "Error",           "Valor": global_m["n_error"]},
    {"Métrica": "MASE mediana",    "Valor": round(global_m["mase_median"], 4)},
    {"Métrica": "MASE media",      "Valor": round(global_m["mase_mean"],   4)},
    {"Métrica": "MASE p75",        "Valor": round(global_m["mase_p75"],    4)},
    {"Métrica": "MASE p90",        "Valor": round(global_m["mase_p90"],    4)},
    {"Métrica": "WAPE mediana",    "Valor": round(global_m["wape_median"], 4)},
    {"Métrica": "Bias mediano",    "Valor": round(global_m["bias_median"], 4)},
    {"Métrica": "Tiempo total (s)","Valor": result.elapsed_seconds},
])
display(summary.reset_index(drop=True))

,Métrica,Valor
0,SKUs evaluados,800.0000
1,OK,746.0000
2,Fallback,37.0000
3,No forecast,0.0000
4,Error,17.0000
5,MASE mediana,0.7271
6,MASE media,0.7720
7,MASE p75,0.9191
8,MASE p90,1.1320
9,WAPE mediana,0.1917


## 3 — Distribución de MASE

In [8]:
valid = df[df["status"].isin(["ok", "fallback"]) & df["mase"].notna()]

fig = px.histogram(
    valid,
    x="mase",
    color="sb_class",
    nbins=50,
    barmode="overlay",
    opacity=0.7,
    title="Distribución de MASE por clase S/B",
    labels={"mase": "MASE", "sb_class": "Clase S/B"},
    template="plotly_white",
)
fig.add_vline(
    x=float(valid["mase"].median()),
    line_dash="dash",
    line_color="black",
    annotation_text=f"Mediana={valid['mase'].median():.3f}",
)
fig.update_layout(height=450)
fig.show()

## 4 — MASE por segmento (heatmap)

In [9]:
# Pivot: sb_class × abc_class → MASE mediana
pivot = (
    valid
    .groupby(["sb_class", "abc_class"])["mase"]
    .median()
    .unstack("abc_class")
)

fig = px.imshow(
    pivot,
    text_auto=".3f",
    color_continuous_scale="RdYlGn_r",
    title="MASE mediana: Clase S/B × Clase ABC",
    labels={"x": "ABC", "y": "S/B", "color": "MASE mediana"},
    template="plotly_white",
)
fig.update_layout(height=350)
fig.show()

## 5 — Distribución de modelos ganadores

In [10]:
ms = aggregator.compute_model_selection_summary(df)

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "pie"}, {"type": "bar"}]],
    subplot_titles=["Global", "Por clase S/B"],
)

fig.add_trace(
    go.Pie(
        labels=ms["model_winner"],
        values=ms["n_skus"],
        textinfo="label+percent",
        hole=0.35,
    ),
    row=1, col=1,
)

ms_seg = aggregator.compute_model_selection_summary(df, by="sb_class")
for model in ms_seg["model_winner"].unique():
    sub = ms_seg[ms_seg["model_winner"] == model]
    fig.add_trace(
        go.Bar(name=model, x=sub["sb_class"], y=sub["pct"], text=sub["pct"].round(2), textposition="auto"),
        row=1, col=2,
    )

fig.update_layout(
    title="Distribución de modelos ganadores (horse-race)",
    barmode="stack",
    template="plotly_white",
    height=420,
    legend_title="Modelo",
)
fig.show()

## 6 — Horse-race: box plots de MASE por modelo

In [11]:
mase_cols = [c for c in df.columns if c.startswith("mase_")]

# Derretir a formato long
horse = (
    valid[mase_cols]
    .melt(var_name="modelo", value_name="mase_backtest")
    .dropna(subset=["mase_backtest"])
)
horse["modelo"] = horse["modelo"].str.replace("mase_", "", regex=False)

fig = px.box(
    horse,
    x="modelo",
    y="mase_backtest",
    color="modelo",
    points="outliers",
    title="MASE backtest por modelo (todos los SKUs donde compitió)",
    labels={"mase_backtest": "MASE", "modelo": "Modelo"},
    template="plotly_white",
)
fig.update_layout(height=450, showlegend=False)
fig.update_yaxes(range=[0, valid["mase"].quantile(0.95) * 2])
fig.show()

## 7 — Bias vs MASE (scatter por SKU)

In [12]:
scatter_df = valid[valid["bias"].notna() & valid["mase"].notna()].copy()
mase_cap = scatter_df["mase"].quantile(0.95)
scatter_df = scatter_df[scatter_df["mase"] <= mase_cap]

fig = px.scatter(
    scatter_df,
    x="bias",
    y="mase",
    color="sb_class",
    symbol="abc_class",
    hover_data=["sku", "model_winner", "n_obs"],
    title="Bias vs MASE por SKU (p95 cap en MASE)",
    labels={"bias": "Bias", "mase": "MASE", "sb_class": "Clase S/B"},
    template="plotly_white",
    opacity=0.65,
)
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.add_hline(y=1, line_dash="dash", line_color="gray", annotation_text="MASE=1")
fig.update_layout(height=500)
fig.show()

## 8 — Peores SKUs (MASE más alto)

In [13]:
worst = (
    valid
    .sort_values("mase", ascending=False)
    .head(20)[
        ["sku", "sb_class", "abc_class", "model_winner", "mase", "wape", "bias", "n_obs"]
    ]
    .reset_index(drop=True)
)
worst.index += 1
worst["mase"] = worst["mase"].round(4)
worst["wape"] = worst["wape"].round(4)
worst["bias"] = worst["bias"].round(4)
display(worst)

,sku,sb_class,abc_class,model_winner,mase,wape,bias,n_obs
1,SKU-00542,lumpy,C,ADIDA,5.5091,1.0001,-0.9997,26.0
2,SKU-00646,intermittent,C,ADIDA,3.3955,1.0314,-0.9059,31.0
3,SKU-00062,smooth,C,SeasonalNaive,2.5684,0.2906,0.2423,36.0
4,SKU-00528,intermittent,C,ADIDA,2.5471,0.9350,-0.9350,31.0
5,SKU-00658,lumpy,C,ADIDA,2.3455,1.0005,-0.9984,22.0
6,SKU-00094,smooth,C,AutoETS,2.2841,0.3642,0.0730,36.0
7,SKU-00376,smooth,C,AutoARIMA,2.2640,0.2656,-0.0911,36.0
8,SKU-00278,intermittent,C,ADIDA,2.2350,0.9940,-0.9821,30.0
9,SKU-00429,intermittent,C,ADIDA,2.0074,1.0002,-0.9995,27.0
10,SKU-00586,smooth,B,AutoARIMA,1.9030,0.3136,0.0554,36.0


## 9 — Comparación entre múltiples runs

Rellena `RUN_IDS` con los IDs de runs que quieras comparar.

In [14]:
# Listar todos los runs disponibles
available = run_store.list_runs(BASE_DIR)
display(available[["run_id", "run_name", "created_at", "n_skus", "n_ok", "mase_global_median", "h", "n_windows", "use_lgbm"]])

,run_id,run_name,created_at,n_skus,n_ok,mase_global_median,h,n_windows,use_lgbm
0,20260325_172913_baseline,baseline,2026-03-25T17:32:09.178893+00:00,800,746,0.7271,3,3,False
1,20260325_172358_baseline,baseline,2026-03-25T17:26:41.234372+00:00,800,746,0.7271,3,3,False
2,20260325_171914_baseline,baseline,2026-03-25T17:24:04.978396+00:00,800,746,0.7271,3,3,False
3,20260325_171715_baseline,baseline,2026-03-25T17:18:06.252209+00:00,800,0,NaN,3,3,False
4,20260325_141406_baseline,baseline,2026-03-25T14:32:55.966249+00:00,800,746,0.7271,3,3,False


In [15]:
# ── Ajustar con los IDs que quieras comparar ─────────────────────────────────
RUN_IDS = available["run_id"].tolist()[:4]  # últimos 4 runs por defecto

if len(RUN_IDS) >= 2:
    comp = comparator.compare_runs(RUN_IDS, base_dir=BASE_DIR, metric="mase")
    display(comp)

    fig = px.bar(
        comp,
        x="run_name",
        y="mase_median",
        error_y=comp["mase_p75"] - comp["mase_median"],
        color="run_name",
        title="MASE mediana por run (barra de error = p75)",
        labels={"mase_median": "MASE mediana", "run_name": "Run"},
        template="plotly_white",
    )
    fig.update_layout(height=400, showlegend=False)
    fig.show()
else:
    print("Se necesitan al menos 2 runs para comparar.")

,run_id,run_name,n_skus,n_ok,n_fallback,n_no_forecast,n_error,mase_median,mase_mean,mase_p75,mase_p90,granularity,h,n_windows,use_lgbm,elapsed_s
0,20260325_172913_baseline,baseline,800,746,37,0,17,0.7271,0.772,0.9191,1.132,M,3,3,False,153.0
1,20260325_172358_baseline,baseline,800,746,37,0,17,0.7271,0.772,0.9191,1.132,M,3,3,False,132.8
2,20260325_171914_baseline,baseline,800,746,37,0,17,0.7271,0.772,0.9191,1.132,M,3,3,False,260.8
3,20260325_171715_baseline,baseline,800,0,0,0,800,NaN,NaN,NaN,NaN,M,3,3,False,30.3


In [16]:
# Comparación por segmento (sb_class)
if len(RUN_IDS) >= 2:
    seg_comp = comparator.compare_runs_by_segment(
        RUN_IDS, segment_col="sb_class", base_dir=BASE_DIR, metric="mase"
    )
    if not seg_comp.empty:
        fig = px.bar(
            seg_comp.reset_index().melt(id_vars="sb_class", var_name="run", value_name="mase_median"),
            x="sb_class",
            y="mase_median",
            color="run",
            barmode="group",
            title="MASE mediana por clase S/B y run",
            labels={"mase_median": "MASE mediana", "sb_class": "Clase S/B"},
            template="plotly_white",
        )
        fig.update_layout(height=400)
        fig.show()

In [17]:
# SKUs donde cambió el modelo ganador entre los dos primeros runs
if len(RUN_IDS) >= 2:
    changes = comparator.find_winner_changes(RUN_IDS[0], RUN_IDS[1], base_dir=BASE_DIR)
    if not changes.empty:
        print(f"{len(changes)} SKUs cambiaron de modelo ganador entre {RUN_IDS[0]} y {RUN_IDS[1]}")
        display(changes.head(20))
    else:
        print("Ningún SKU cambió de modelo ganador entre los dos runs.")

Ningún SKU cambió de modelo ganador entre los dos runs.
